# H-FM10 — NV-Segment-CT pretrained fine-tuning for CT hemorrhage

CT 뇌출혈 binary segmentation foundation-model baseline이다. 기존 Dataset001 fold-0 split(train 186 / validation 40)을 그대로 사용한다.

- backbone: NVIDIA NV-Segment-CT automatic branch
- 방식: fixed-channel softmax fine-tuning
- mapping: dataset hemorrhage label `1` → pretrained VISTA `brain` class ID `22`
- 주의: brain ID는 출혈의 zero-shot class가 아니다. CT anatomy 기반 initialization 후 출혈 label로 full fine-tuning하는 baseline이다.

공식 softmax recipe는 train/inference에 동일한 label mapping 순서를 요구한다.




In [13]:
from pathlib import Path
import json, subprocess, shutil

REPO = Path("/home/stu03/workspace/model")

# 기존 프로젝트 Python 3.12
PYTHON = REPO / ".venv/bin/python"

CONFIG = REPO / "configs/experiments/hemorrhage/H-FM10.yaml"

BUNDLE = REPO / "external/NV-Segment-CTMR/NV-Segment-CT"

# NV-Segment-CT / CTMR 공용 Python 3.10 환경
NV_VENV = REPO / "external/.venv-nv"
NV_PYTHON = NV_VENV / "bin/python"

OVERRIDE = REPO / "configs/nvsegmentct/H-FM10_train_override.json"
RUN = REPO / "runs/H-FM10"

settings = json.loads(
    json.dumps(
        __import__("yaml").safe_load(CONFIG.read_text())
    )
)

assert BUNDLE.is_dir(), \
    "Official bundle missing: clone external/NV-Segment-CTMR first"

assert NV_PYTHON.exists(), \
    f"NV Python not found: {NV_PYTHON}"

print(settings["experiment"])
print("bundle:", BUNDLE)
print("NV venv:", NV_VENV)
print("NV python:", NV_PYTHON)
print("run:", RUN)

{'name': 'H-FM10', 'family': 'NV-Segment-CT', 'task': 'CT intracranial hemorrhage binary segmentation', 'split': 'Dataset001_ICHBinary fold 0 (train=186, val=40)'}
bundle: /home/stu03/workspace/model/external/NV-Segment-CTMR/NV-Segment-CT
NV venv: /home/stu03/workspace/model/external/.venv-nv
NV python: /home/stu03/workspace/model/external/.venv-nv/bin/python
run: /home/stu03/workspace/model/runs/H-FM10


## Fold-0 datalist 생성




In [14]:
data=settings["data"]
subprocess.run([
    str(PYTHON),str(REPO/"scripts/hemorrhage/prepare_h_fm10_nvsegmentct_data.py"),
    "--dataset-root",data["nnunet_raw"],"--splits",data["splits"],"--output",data["datalist"],
],check=True,cwd=REPO)
datalist=json.loads(Path(data["datalist"]).read_text())
print("records:",len(datalist["training"]),"validation:",sum(x["fold"]==0 for x in datalist["training"]))




{"train": 186, "val": 40, "output": "/home/stu03/workspace/model/runs/H-FM10/data/datalist_fold0.json"}
records: 226 validation: 40


## 사전학습 checkpoint 준비

처음 한 번만 실행한다. Hugging Face에서 `nvidia/NV-Segment-CT` checkpoint를 다운로드하므로 네트워크와 저장 공간이 필요하다.




In [16]:
import sys

print("Notebook Python:", sys.executable)
print("NV Python:", NV_PYTHON)
assert NV_PYTHON.exists(), f"NV Python not found: {NV_PYTHON}"
subprocess.run([str(NV_PYTHON), "--version"], check=True)

# NV-Segment-CT's official requirements pin torch==2.0.1, which cannot run on RTX 5090
# (compute capability sm_120). Install every other dependency first.
requirements = BUNDLE / "requirements.txt"
filtered_requirements = RUN / "configs/nvsegmentct_requirements_without_torch.txt"
filtered_requirements.parent.mkdir(parents=True, exist_ok=True)
filtered_requirements.write_text("\n".join(
    line for line in requirements.read_text().splitlines()
    if not line.strip().lower().startswith(("torch", "torchvision", "torchaudio"))
) + "\n")

subprocess.run([str(NV_PYTHON), "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([str(NV_PYTHON), "-m", "pip", "install", "-r", str(filtered_requirements)], check=True)

CHECKPOINT = BUNDLE / "models/model.pt"
if not CHECKPOINT.exists():
    subprocess.run([
        str(NV_PYTHON), "-c",
        "from scripts.huggingface_download import prepare_huggingface_checkpoint; "
        "prepare_huggingface_checkpoint('nvidia/NV-Segment-CT', "
        "'vista3d_pretrained_model/model.pt', 'models/model.pt')",
    ], cwd=BUNDLE, check=True)
else:
    print("Checkpoint already exists:", CHECKPOINT)


Notebook Python : /home/stu03/workspace/model/.venv/bin/python
Notebook version: 3.12.13 (main, Jul 14 2026, 02:09:00) [GCC 14.2.0]
NV Python       : /home/stu03/workspace/model/external/.venv-nv/bin/python
NV Python version: Python 3.10.21
  Using cached matplotlib-3.8.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.8 kB)
  Using cached monai-1.4.0-py3-none-any.whl.metadata (11 kB)
  Using cached scikit_image-0.24.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (14 kB)
  Using cached torch-2.0.1-cp310-cp310-manylinux1_x86_64.whl.metadata (24 kB)
  Using cached tensorboard-2.13.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached timm-1.0.29-py3-none-any.whl.metadata (43 kB)
  Using cached pytorch_ignite-0.5.5-py3-none-any.whl.metadata (28 kB)
  Using cached mlflow-3.16.1-py3-none-any.whl.metadata (50 kB)
  Using cached huggingface_hub-1.32.0-py3-none-any.whl.metadata (16 kB)
  Using cached nvidia_cudnn_cu11-8.5.0.96-2-py3-none-manylinux1_x

[nvseg] registered Hugging Face download for nvidia/NV-Segment-CT/config.json
[nvseg] prepared checkpoint at models/model.pt

NV python : /home/stu03/workspace/model/external/.venv-nv/bin/python
Checkpoint: /home/stu03/workspace/model/external/NV-Segment-CTMR/NV-Segment-CT/models/model.pt


In [23]:
# RTX 5090 (Blackwell, sm_120) requires a CUDA 12.8+ PyTorch wheel.
# Run once after dependency installation, and again only if torch is later overwritten.
subprocess.run([
    str(NV_PYTHON), "-m", "pip", "install", "--upgrade", "--force-reinstall", "--no-cache-dir",
    "torch==2.7.0", "torchvision==0.22.0", "torchaudio==2.7.0",
    "--index-url", "https://download.pytorch.org/whl/cu128",
], check=True)

# MONAI/Nibabel compatibility used by the official bundle.
subprocess.run([
    str(NV_PYTHON), "-m", "pip", "install", "--upgrade", "--force-reinstall", "numpy==1.26.4"
], check=True)
print("Installed CUDA 12.8 PyTorch stack for RTX 5090.")


Found existing installation: torch 2.0.1
Uninstalling torch-2.0.1:
  Successfully uninstalled torch-2.0.1
Looking in indexes: https://download.pytorch.org/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 39.7 MB/s  0:00:02 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 10.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 43.6 MB/s  0:00:00eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 726.9/726.9 MB 45.8 MB/s  0:00:16:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 609.6/609.6 MB 47.7 MB/s  0:00:13:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 53.0 MB/s  0:00:03:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 54.8 MB/s  0:00:01 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.4/260.4 MB 48.7 MB/s  0:00:05:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.1/292.1 MB 44.3 MB/s  0:00:06:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.15.2 requires torch==2.0.1, but you have torch 2.7.0+cu128 which is incompatible.


CompletedProcess(args=['/home/stu03/workspace/model/external/.venv-nv/bin/python', '-m', 'pip', 'install', '--no-cache-dir', 'torch==2.7.0', '--index-url', 'https://download.pytorch.org/whl/cu128'], returncode=0)

In [24]:
result = subprocess.run([
    str(NV_PYTHON), "-c", """
import torch
assert torch.cuda.is_available(), "CUDA is not available in the NV environment"
capability = torch.cuda.get_device_capability(0)
archs = torch.cuda.get_arch_list()
print("Torch        :", torch.__version__)
print("CUDA runtime :", torch.version.cuda)
print("GPU          :", torch.cuda.get_device_name(0))
print("Capability   :", capability)
print("Arch list    :", archs)
# RTX 5090 requires Blackwell support. The actual GPU operation is the final check.
x = torch.arange(10, device="cuda")
print("GPU tensor   :", x)
print("CUDA TEST OK")
"""
], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError("NV CUDA environment check failed; rerun the preceding CUDA 12.8 PyTorch setup cell.")


Torch        : 2.7.0+cu128
CUDA runtime : 12.8
CUDA usable  : True
GPU          : NVIDIA GeForce RTX 5090
Capability   : (12, 0)
Arch list    : ['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120', 'compute_120']
GPU tensor   : tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], device='cuda:0')
CUDA TEST OK



## 5-epoch smoke fine-tune

정식 학습 전, datalist·checkpoint·GPU·메모리만 검증한다. 성공 후 정식 셀을 실행한다.




In [29]:
import json
import subprocess
from pathlib import Path

smoke_override = RUN / "configs/H-FM10_smoke_override.json"

smoke = json.loads(OVERRIDE.read_text())

# --------------------------------------------------
# 중요: 기존 OVERRIDE에 들어 있는 train / validate 전체 override 제거
# --------------------------------------------------

smoke.pop("train", None)
smoke.pop("validate", None)

# --------------------------------------------------
# Smoke test 전용 override
# --------------------------------------------------

smoke.update({
    "epochs": 5,

    "n_train_samples": 4,
    "n_val_samples": 2,

    "patch_size": [96, 96, 64],
    "patch_size_valid": [96, 96, 64],

    "train_dataset_cache_rate": 0.0,
    "val_dataset_cache_rate": 0.0,
    "show_cache_progress": False,

    "use_tensorboard": False,
    "use_mlflow": False,

    # train / validate 전체를 덮지 않고 필요한 항목만 변경
    "train#dataloader#num_workers": 0,
    "train#dataloader#persistent_workers": False,
    "validate#dataloader#num_workers": 0,
})

smoke_override.parent.mkdir(
    parents=True,
    exist_ok=True
)

smoke_override.write_text(
    json.dumps(smoke, indent=2) + "\n"
)

# --------------------------------------------------
# 확인: train / validate top-level key가 없어야 함
# --------------------------------------------------

print("Has top-level train?   ", "train" in smoke)
print("Has top-level validate?", "validate" in smoke)

assert "train" not in smoke
assert "validate" not in smoke

# --------------------------------------------------
# MONAI Bundle 실행
# --------------------------------------------------

config_arg = (
    f"['configs/train_continual_softmax.json',"
    f"'{smoke_override}']"
)

command = [
    str(NV_PYTHON),
    "-m",
    "monai.bundle",
    "run",
    "--config_file",
    config_arg,
]

print("\nCOMMAND:")
print(" ".join(command))

print("\nNV Python:")
subprocess.run([
    str(NV_PYTHON),
    "--version"
])

print("\nSmoke overrides:")
print(json.dumps(smoke, indent=2))

result = subprocess.run(
    command,
    cwd=BUNDLE,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(result.stdout)

if result.returncode != 0:
    raise RuntimeError(
        f"NV-Segment-CT smoke test failed "
        f"(exit code={result.returncode})"
    )

print("Smoke test completed successfully.")

Has top-level train?    False
Has top-level validate? False

COMMAND:
/home/stu03/workspace/model/external/.venv-nv/bin/python -m monai.bundle run --config_file ['configs/train_continual_softmax.json','/home/stu03/workspace/model/runs/H-FM10/configs/H-FM10_smoke_override.json']

NV Python:
Python 3.10.21

Smoke overrides:
{
  "bundle_root": "/home/stu03/workspace/model/external/NV-Segment-CTMR/NV-Segment-CT",
  "ckpt_dir": "/home/stu03/workspace/model/runs/H-FM10/bundle/models",
  "output_dir": "/home/stu03/workspace/model/runs/H-FM10/bundle/eval",
  "mlflow_dir": "/home/stu03/workspace/model/runs/H-FM10/bundle/mlruns",
  "data_list_file_path": "/home/stu03/workspace/model/runs/H-FM10/data/datalist_fold0.json",
  "dataset_dir": "/",
  "finetune_model_path": "/home/stu03/workspace/model/external/NV-Segment-CTMR/NV-Segment-CT/models/model.pt",
  "label_mappings": {
    "default": [
      [
        1,
        22
      ]
    ]
  },
  "epochs": 5,
  "val_interval": 1,
  "val_at_start": true

In [31]:
import json
import shutil
from pathlib import Path

# 현재 OVERRIDE 읽기
cfg = json.loads(OVERRIDE.read_text())

print("Before:")
print("train:", cfg.get("train"))
print("validate:", cfg.get("validate"))

# --------------------------------------------------
# 안전장치: 우리가 예상한 dataloader 설정만 들어있는지 확인
# --------------------------------------------------

train_cfg = cfg.get("train", {})
validate_cfg = cfg.get("validate", {})

if train_cfg:
    unexpected = set(train_cfg.keys()) - {"dataloader"}
    assert not unexpected, (
        f"train 안에 dataloader 외 설정이 있습니다: {unexpected}\n"
        "자동 삭제하지 않고 중단합니다."
    )

if validate_cfg:
    unexpected = set(validate_cfg.keys()) - {"dataloader"}
    assert not unexpected, (
        f"validate 안에 dataloader 외 설정이 있습니다: {unexpected}\n"
        "자동 삭제하지 않고 중단합니다."
    )

# --------------------------------------------------
# 기존 dataloader 값을 path-style override로 이동
# --------------------------------------------------

train_dl = train_cfg.get("dataloader", {})
validate_dl = validate_cfg.get("dataloader", {})

if "num_workers" in train_dl:
    cfg["train#dataloader#num_workers"] = train_dl["num_workers"]

if "persistent_workers" in train_dl:
    cfg["train#dataloader#persistent_workers"] = train_dl["persistent_workers"]

if "num_workers" in validate_dl:
    cfg["validate#dataloader#num_workers"] = validate_dl["num_workers"]

# --------------------------------------------------
# 문제를 일으키는 top-level 블록 제거
# --------------------------------------------------

cfg.pop("train", None)
cfg.pop("validate", None)

# --------------------------------------------------
# 원본 백업
# --------------------------------------------------

backup = OVERRIDE.with_suffix(".json.bak")

if not backup.exists():
    shutil.copy2(OVERRIDE, backup)

# 수정본 저장
OVERRIDE.write_text(
    json.dumps(cfg, indent=2) + "\n"
)

print("\nAfter:")
print("train exists?   ", "train" in cfg)
print("validate exists?", "validate" in cfg)

print("train workers:",
      cfg.get("train#dataloader#num_workers"))
print("train persistent:",
      cfg.get("train#dataloader#persistent_workers"))
print("val workers:",
      cfg.get("validate#dataloader#num_workers"))

print("\nSaved:", OVERRIDE)
print("Backup:", backup)

Before:
train: {'dataloader': {'num_workers': 2, 'persistent_workers': True}}
validate: {'dataloader': {'num_workers': 2}}

After:
train exists?    False
validate exists? False
train workers: 2
train persistent: True
val workers: 2

Saved: /home/stu03/workspace/model/configs/nvsegmentct/H-FM10_train_override.json
Backup: /home/stu03/workspace/model/configs/nvsegmentct/H-FM10_train_override.json.bak


## 정식 fine-tuning

checkpoint 준비와 smoke 성공 후 주석을 해제한다. 결과는 `runs/H-FM10/bundle/`에 저장된다.




In [33]:
import json
import subprocess

# --------------------------------------------------
# 1. Full-training override 검증
# --------------------------------------------------

override_cfg = json.loads(OVERRIDE.read_text())

assert "train" not in override_cfg, (
    "OVERRIDE에 top-level 'train'이 있습니다. "
    "train#dataloader#... 형식으로 바꿔야 합니다."
)

assert "validate" not in override_cfg, (
    "OVERRIDE에 top-level 'validate'가 있습니다. "
    "validate#dataloader#... 형식으로 바꿔야 합니다."
)

print("Override config OK")
print("config:", OVERRIDE)


# --------------------------------------------------
# 2. H-FM10 full training
# --------------------------------------------------

command = [
    str(NV_PYTHON),
    "-m",
    "monai.bundle",
    "run",
    "--config_file",
    f"['configs/train_continual_softmax.json','{OVERRIDE}']"
]

print("\nCOMMAND:")
print(" ".join(command))


# --------------------------------------------------
# 3. 실행
# --------------------------------------------------

subprocess.run(
    command,
    cwd=BUNDLE,
    check=True
)

Override config OK
config: /home/stu03/workspace/model/configs/nvsegmentct/H-FM10_train_override.json

COMMAND:
/home/stu03/workspace/model/external/.venv-nv/bin/python -m monai.bundle run --config_file ['configs/train_continual_softmax.json','/home/stu03/workspace/model/configs/nvsegmentct/H-FM10_train_override.json']
2026-09-21 02:07:43,562 - INFO - --- input summary of monai.bundle.scripts.run ---
2026-09-21 02:07:43,562 - INFO - > config_file: ['configs/train_continual_softmax.json',
 '/home/stu03/workspace/model/configs/nvsegmentct/H-FM10_train_override.json']
2026-09-21 02:07:43,562 - INFO - ---


2026-09-21 02:07:43,562 - WARNING - Not all config files are in configs. If logging_file and meta_file arenot specified, configs will be used as the default config root directory.
2026-09-21 02:07:43,562 - INFO - Setting logging properties based on config: configs/logging.conf.


/home/stu03/workspace/model/external/.venv-nv/lib/python3.10/site-packages/monai/engines/trainer.py:54: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler() if self.amp else None


2026-09-21 02:07:44,272 - ignite.engine.engine.SupervisedTrainer - INFO - Engine run resuming from iteration 0, epoch 0 until 150 epochs
2026-09-21 02:07:44,486 - ignite.engine.engine.SupervisedTrainer - INFO - Restored all variables from /home/stu03/workspace/model/external/NV-Segment-CTMR/NV-Segment-CT/models/model.pt
2026-09-21 02:07:44,486 - ignite.engine.engine.SupervisedEvaluator - INFO - Engine run resuming from iteration 0, epoch -1 until 1 epochs


/home/stu03/workspace/model/external/.venv-nv/lib/python3.10/site-packages/monai/engines/evaluator.py:331: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(**engine.amp_kwargs):


2026-09-21 02:08:13,193 - ignite.engine.engine.SupervisedEvaluator - INFO - Got new best metric of val_mean_dice: 0.035614848136901855
2026-09-21 02:08:13,194 - INFO - Epoch[0] Metrics -- val_mean_dice: 0.0356 
2026-09-21 02:08:13,194 - INFO - Key metric: val_mean_dice best value: 0.035614848136901855 at epoch: 0
2026-09-21 02:08:13,683 - ignite.engine.engine.SupervisedEvaluator - INFO - Epoch[0] Complete. Time taken: 00:00:29.197
2026-09-21 02:08:13,683 - ignite.engine.engine.SupervisedEvaluator - INFO - Engine run finished. Time taken: 00:00:29.197


/home/stu03/workspace/model/external/.venv-nv/lib/python3.10/site-packages/monai/engines/trainer.py:257: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(**engine.amp_kwargs):


2026-09-21 02:08:14,396 - INFO - Epoch: 1/150, Iter: 1/186 -- train_loss: 1.5255 
2026-09-21 02:08:14,893 - INFO - Epoch: 1/150, Iter: 2/186 -- train_loss: 2.3273 
2026-09-21 02:08:15,393 - INFO - Epoch: 1/150, Iter: 3/186 -- train_loss: 1.8544 
2026-09-21 02:08:15,810 - INFO - Epoch: 1/150, Iter: 4/186 -- train_loss: 1.2428 
2026-09-21 02:08:16,369 - INFO - Epoch: 1/150, Iter: 5/186 -- train_loss: 1.3026 
2026-09-21 02:08:16,843 - INFO - Epoch: 1/150, Iter: 6/186 -- train_loss: 1.3357 
2026-09-21 02:08:17,307 - INFO - Epoch: 1/150, Iter: 7/186 -- train_loss: 1.2227 
2026-09-21 02:08:17,798 - INFO - Epoch: 1/150, Iter: 8/186 -- train_loss: 1.0528 
2026-09-21 02:08:18,247 - INFO - Epoch: 1/150, Iter: 9/186 -- train_loss: 1.0738 
2026-09-21 02:08:18,808 - INFO - Epoch: 1/150, Iter: 10/186 -- train_loss: 1.0025 
2026-09-21 02:08:19,273 - INFO - Epoch: 1/150, Iter: 11/186 -- train_loss: 1.0421 
2026-09-21 02:08:19,762 - INFO - Epoch: 1/150, Iter: 12/186 -- train_loss: 1.0123 
2026-09-21 02

CompletedProcess(args=['/home/stu03/workspace/model/external/.venv-nv/bin/python', '-m', 'monai.bundle', 'run', '--config_file', "['configs/train_continual_softmax.json','/home/stu03/workspace/model/configs/nvsegmentct/H-FM10_train_override.json']"], returncode=0)

## 다음 단계

학습 후 official inference를 40 validation case에 수행해 원본 NIfTI 공간 mask/probability를 저장하고, 기존 H-N 모델과 같은 Dice·lesion recall/F2·FP/scan·threshold sweep 평가를 붙인다. 이 평가 export는 checkpoint가 실제 생성된 뒤 구성한다.




## 40-case validation export and threshold evaluation

`model_softmax.pt`가 생성된 뒤 실행한다. 공식 softmax model로 fold-0 validation 40건을 추론하고, **원본 NIfTI 공간** probability map과 threshold별 mask를 저장한다. 이후 Dice, voxel recall/precision, lesion recall/precision/F2, FP/scan threshold sweep을 생성한다.


In [ ]:
FINAL = RUN / "final_metrics"
FINETUNED_CHECKPOINT = RUN / "bundle/models/model_softmax.pt"
DATALIST = Path(settings["data"]["datalist"])

assert FINETUNED_CHECKPOINT.is_file(), f"Fine-tuned checkpoint missing: {FINETUNED_CHECKPOINT}"
assert DATALIST.is_file(), f"Datalist missing: {DATALIST}"

command = [
    str(NV_PYTHON), str(REPO / "scripts/hemorrhage/evaluate_h_fm10.py"),
    "--bundle-dir", str(BUNDLE),
    "--checkpoint", str(FINETUNED_CHECKPOINT),
    "--datalist", str(DATALIST),
    "--output-dir", str(FINAL),
]
print(" ".join(command))
subprocess.run(command, cwd=REPO, check=True)


In [ ]:
import pandas as pd
from IPython.display import display

sweep_path = FINAL / "threshold_sweep.csv"
summary_path = FINAL / "summary.json"
assert sweep_path.is_file(), "Evaluation has not completed: threshold_sweep.csv is missing."

sweep = pd.read_csv(sweep_path)
summary = json.loads(summary_path.read_text())
display(sweep[[
    "threshold", "macro_dice", "voxel_recall", "voxel_precision",
    "lesion_recall", "lesion_precision", "lesion_f2", "fp_per_scan",
]].sort_values("lesion_f2", ascending=False).head(10))
print("Recommended threshold (lesion F2 with FP/scan ≤ 1.5):")
print(json.dumps(summary["recommended_lesion_f2"], indent=2))
print("Probability maps and threshold masks:", FINAL / "probabilities")
